# Time Series Prediction for Predictive Maintenance

This notebook keeps the original code unchanged and only separates it into notebook-style sections.  
Each markdown cell briefly explains what the following code block is doing.  
The flow follows the paper’s TSP approach: data preparation, ARIMA model selection, forecasting, and visualization.

### 1) Imports

This block loads the numerical, plotting, and time-series libraries used by the script.  
It also suppresses warnings so repeated model fitting does not flood the notebook output.  
These imports support the ADF, ACF/PACF, Ljung-Box, and ARIMA steps later on.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox



### 2) Global parameters

This block defines the sick and dead specification thresholds, the model window size, and the forecast horizon.  
It also fixes the random seed so the synthetic degradation trajectory is reproducible.  
These values control how the later TSP logic behaves.

In [ ]:
# =============================================================================
# 0. PARAMETERS
# =============================================================================
np.random.seed(42)

SICK_SPEC = 7.5
DEAD_SPEC = 10.5
M_MODEL = 30
FORECAST_HORIZON = 250



### 3) Synthetic degradation data

This block constructs a throttle-valve-like time series with four phases: healthy growth, flat sick behavior, a sudden rise, and near-dead oscillation.  
The piecewise construction is meant to mimic the kind of aging-feature trend discussed in the paper.  
The code then stores the sick-state index for later forecasting.

In [ ]:
# =============================================================================
# 1. SYNTHESISE THROTTLE-VALVE DATA
# =============================================================================
pre_n = 30
total_n = pre_n + 109
spans = np.arange(267, 267 + total_n)
DEATH_SPAN = int(spans[-1])

YT = np.zeros(total_n)

# Phase 1 – pre-sick
for i in range(pre_n):
    YT[i] = 5.0 + 0.008 * i + np.random.normal(0, 0.06)

# Phase 2 – flat/sick
flat_end = pre_n + 82
for i in range(pre_n, flat_end):
    YT[i] = YT[pre_n - 1] + 0.012 * (i - pre_n) + np.random.normal(0, 0.10)

# Phase 3 – sudden rise
rise_n = 6
for i in range(flat_end, flat_end + rise_n):
    step = i - flat_end + 1
    YT[i] = YT[flat_end - 1] + step * 1.90 + np.random.normal(0, 0.30)

# Phase 4 – oscillation near dead spec
osc_start = flat_end + rise_n
for i in range(osc_start, total_n):
    rel = i - osc_start
    YT[i] = 12.8 + 2.6 * np.sin(0.9 * rel) + 0.08 * rel + np.random.normal(0, 0.5)
    YT[i] = max(YT[i], DEAD_SPEC - 0.4)

SICK_IDX = pre_n



### 4) TSP model creation

This block pulls the pre-sick samples and decides whether to log-transform them.  
It checks stationarity with ADF, inspects ACF/PACF, tests for white-noise behavior, and then searches ARIMA orders by BIC.  
The best order is saved so the same structure can be reused during rolling prediction.

In [ ]:
# =============================================================================
# 2. TSP MODEL CREATION
# =============================================================================
YM = YT[:SICK_IDX].copy()

# Log transform if variance grows
h = M_MODEL // 2
use_log = (np.var(YM[h:]) > 2.0 * np.var(YM[:h])) and np.all(YM > 0)
YM_w = np.log(YM) if use_log else YM.copy()
print(f"[Step 5] Log transform applied: {use_log}")

# ADF test
adf_p = adfuller(YM_w)[1]
is_stat = adf_p < 0.05
print(f"[Step 6] ADF p-value: {adf_p:.4f} | Stationary: {is_stat}")

d_order = 0
if not is_stat:
    d_order = 1
    print("[Step 7] Applying 1st-order differencing")

# ACF / PACF bounds
max_lag = min(11, M_MODEL // 3)
acf_v = np.abs(acf(YM_w, nlags=max_lag, fft=False)[1:])
pacf_v = np.abs(pacf(YM_w, nlags=max_lag, method="ols")[1:])
A = min(int(np.argmax(acf_v)) + 1, 10)
B = min(int(np.argmax(pacf_v)) + 1, 9)
print(f"[Step 8] ACF max-lag A={A} | PACF max-lag B={B}")

# Ljung-Box test
lb_p = acorr_ljungbox(YM_w, lags=[min(10, M_MODEL // 4)], return_df=True)["lb_pvalue"].values[0]
is_wn = lb_p > 0.05
print(f"[Step 9] Ljung-Box p-value: {lb_p:.4f} | White-noise: {is_wn}")
if is_wn:
    A, B = 1, 1

# BIC grid search
print("[Step 11-13] Searching ARIMA combinations by BIC ...")
best_bic, best_order, best_result = np.inf, (1, d_order, 0), None

for p in range(1, min(B + 1, 6)):
    for q in range(0, min(A + 1, 5)):
        try:
            res = ARIMA(YM_w, order=(p, d_order, q)).fit(method_kwargs={"warn_convergence": False})
            if res.bic < best_bic:
                best_bic, best_order, best_result = res.bic, (p, d_order, q), res
        except Exception:
            pass

print(f"Best order: ARIMA{best_order} | BIC = {best_bic:.3f}")

TSP_ORDER = best_order
USE_LOG = use_log
print(f"Final TSP model: ARIMA{TSP_ORDER}")



### 5) Exponential baseline and rolling forecast

This block defines a simple exponential fit as the comparison baseline.  
It then steps through the sick region, refits the selected ARIMA model on the available history, and forecasts the next value.  
Both the TSP estimate and the exponential estimate are stored for the final comparison plot.

In [ ]:
# =============================================================================
# 3. EXPONENTIAL BASELINE + TSP FORECAST
# =============================================================================
def fit_exp(t_arr, y_arr):
    """Fit F·exp(C·t) to (t, y) pairs."""
    from scipy.optimize import curve_fit

    def fn(t, F, C):
        return F * np.exp(C * t)

    try:
        t_n = t_arr - t_arr[0]
        p, _ = curve_fit(fn, t_n, y_arr, p0=[y_arr[0], 0.001],
                         maxfev=6000, bounds=([0, -1], [1e6, 1]))
        return p
    except Exception:
        return np.array([y_arr[0], 0.001])


TSP_YT = np.full(total_n, np.nan)
EXP_YT = np.full(total_n, np.nan)

for t in range(SICK_IDX, total_n):
    avail = YT[:t + 1]

    # TSP forecast
    try:
        seq = np.log(avail) if USE_LOG else avail.copy()
        seq = seq[-60:] if len(seq) > 60 else seq
        res = ARIMA(seq, order=TSP_ORDER).fit(method_kwargs={"warn_convergence": False})
        fc = res.forecast(steps=FORECAST_HORIZON)
        if USE_LOG:
            fc = np.exp(fc)
        TSP_YT[t] = fc[0]
    except Exception:
        TSP_YT[t] = np.nan

    # Exponential baseline forecast
    n_sick = t - SICK_IDX + 1
    if n_sick >= 5:
        exp_F, exp_C = fit_exp(spans[SICK_IDX:t + 1].astype(float),
                               YT[SICK_IDX:t + 1])
    else:
        exp_F, exp_C = YT[SICK_IDX], 0.001

    t_rel = float(spans[t] - spans[SICK_IDX])
    EXP_YT[t] = exp_F * np.exp(exp_C * (t_rel + 1))




### 6) Final plot

This block filters the visible span and draws the final comparison figure.  
It shows the actual series, the TSP estimate, the exponential estimate, and the sick/dead threshold lines.  
The plot is the end-to-end result of the predictive-maintenance implementation.

In [ ]:
# =============================================================================
# 4. SINGLE PLOT ONLY
# =============================================================================
VIS = spans >= 300

fig, ax = plt.subplots(1, 1, figsize=(15, 4.5))
ax.set_facecolor("#fdfdfd")

ax.plot(spans[VIS], YT[VIS], "r*", ms=4.5, label="Actual", zorder=6)
ax.plot(spans[VIS], TSP_YT[VIS], color="deeppink", lw=1.8, label="TSP-estimated", zorder=5)
ax.plot(spans[VIS], EXP_YT[VIS], color="navy", lw=1.8, ls="--", label="Exponential-estimated", zorder=4)

ax.axhline(SICK_SPEC, color="gold", lw=1.5, ls="--", label=f"Sick Spec ({SICK_SPEC})")
ax.axhline(DEAD_SPEC, color="darkorange", lw=2, ls="-", label=f"Dead Spec ({DEAD_SPEC})")

ax.set_ylabel("$Y_T$")
ax.set_xlabel("Time / span")
ax.set_ylim(0, 22)
ax.grid(alpha=0.25)
ax.legend(loc="upper left", fontsize=8, ncol=3, framealpha=0.85)
ax.set_title("TSP Aging-Feature Prediction")
plt.tight_layout()
plt.show()